In [1]:
!pyspark --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/06 11:21:59 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/11/06 11:21:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 21.0.8
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator

In [ ]:
spark = SparkSession.builder \
    .appName("FlightDelay") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/06 11:22:03 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/11/06 11:22:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/06 11:22:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59699)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/opt/anaconda3/envs/flight_delay/lib/python3.13/site-packages/pyspark/accumu

In [4]:
data_path = "/Users/drashi/Documents/UMBC-DATA606-Capstone/Data/clean_flights.parquet"
df = spark.read.parquet(data_path)


In [5]:
print(f"Loaded dataset from: {data_path}")
print(f"Rows: {df.count():,} | Columns: {len(df.columns)}")
df.show(5, truncate=False)

Loaded dataset from: /Users/drashi/Documents/UMBC-DATA606-Capstone/Data/clean_flights.parquet
Rows: 17,094,473 | Columns: 38


25/11/06 11:22:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+--------------+-----------+----+--------------+---------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+--------------+-----------------+-------+-------+--------+-------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Flight_Number_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName  |DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|DepDel15|DepartureDelayGroups|DepTimeBlk|TaxiOut|WheelsOff|WheelsOn|TaxiIn|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|ArrDel15|ArrivalDelayGroups|ArrTimeBlk|CRSElapsedTime|ActualElapsedTime|AirTime|Flights|Distance|DistanceGroup|
+----+-------+-----+----------+---------+----------+-----------------+-------------------------------+------+---------

### FEATURE ENGINEERING

#### 1.1 Select modeling columns + derive hour + binary target

In [6]:
# keep only features available before departure + our targets
df = df.withColumn("DepHour", (F.col("CRSDepTime")/100).cast("int"))
df = df.withColumn("IsDelayed", F.when(F.col("DepDelayMinutes") > 15, 1).otherwise(0))

use_cols = [
    "Reporting_Airline","Origin","Dest",
    "Month","DayOfWeek","DepHour","Distance",
    "CRSElapsedTime","ActualElapsedTime","AirTime",
    "DepDelayMinutes","IsDelayed"
]
df_model = df.select(use_cols)

print(f"model cols: {len(use_cols)}")
df_model.show(5, truncate=False)

model cols: 12
+-----------------+------+----+-----+---------+-------+--------+--------------+-----------------+-------+---------------+---------+
|Reporting_Airline|Origin|Dest|Month|DayOfWeek|DepHour|Distance|CRSElapsedTime|ActualElapsedTime|AirTime|DepDelayMinutes|IsDelayed|
+-----------------+------+----+-----+---------+-------+--------+--------------+-----------------+-------+---------------+---------+
|YX               |LGA   |CAE |11   |1        |14     |617.0   |134.0         |138.0            |109.0  |0.0            |0        |
|YX               |LGA   |CAE |11   |2        |14     |617.0   |134.0         |116.0            |97.0   |0.0            |0        |
|YX               |LGA   |CAE |11   |3        |14     |617.0   |134.0         |118.0            |96.0   |0.0            |0        |
|YX               |LGA   |CAE |11   |4        |14     |617.0   |134.0         |115.0            |98.0   |0.0            |0        |
|YX               |BOS   |CVG |11   |7        |6      |752.0 

#### 1.2 Reduce airport cardinality (Top-50 + “Other”)

In [7]:
# large categorical cardinality hurts tree models and slows indexing
top_origins = [r["Origin"] for r in df_model.groupBy("Origin").count().orderBy(F.desc("count")).limit(50).collect()]
top_dests   = [r["Dest"]   for r in df_model.groupBy("Dest").count().orderBy(F.desc("count")).limit(50).collect()]

df_simpl = (
    df_model
    .withColumn("Origin_S", F.when(F.col("Origin").isin(top_origins), F.col("Origin")).otherwise(F.lit("Other")))
    .withColumn("Dest_S",   F.when(F.col("Dest").isin(top_dests),   F.col("Dest")).otherwise(F.lit("Other")))
    .drop("Origin","Dest")
)

print("origins:", df_model.select("Origin").distinct().count(), "→", df_simpl.select("Origin_S").distinct().count())
print("dests  :", df_model.select("Dest").distinct().count(),   "→", df_simpl.select("Dest_S").distinct().count())
df_simpl.select("Reporting_Airline","Origin_S","Dest_S").show(5, truncate=False)

origins: 358 → 51
dests  : 358 → 51
+-----------------+--------+------+
|Reporting_Airline|Origin_S|Dest_S|
+-----------------+--------+------+
|YX               |LGA     |Other |
|YX               |LGA     |Other |
|YX               |LGA     |Other |
|YX               |LGA     |Other |
|YX               |BOS     |CVG   |
+-----------------+--------+------+
only showing top 5 rows


#### 1.3 Index categoricals + assemble vector

In [8]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

idx_airline = StringIndexer(inputCol="Reporting_Airline", outputCol="AirlineIndex", handleInvalid="keep")
idx_origin  = StringIndexer(inputCol="Origin_S",          outputCol="OriginIndex",  handleInvalid="keep")
idx_dest    = StringIndexer(inputCol="Dest_S",            outputCol="DestIndex",    handleInvalid="keep")

feature_cols = [
    "AirlineIndex","OriginIndex","DestIndex",
    "Month","DayOfWeek","DepHour","Distance",
    "CRSElapsedTime","ActualElapsedTime","AirTime"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
pp = Pipeline(stages=[idx_airline, idx_origin, idx_dest, assembler])

df_ready = pp.fit(df_simpl).transform(df_simpl)\
    .select("features","DepDelayMinutes","IsDelayed")

df_ready.show(5, truncate=False)

+----------------------------------------------------+---------------+---------+
|features                                            |DepDelayMinutes|IsDelayed|
+----------------------------------------------------+---------------+---------+
|[5.0,11.0,0.0,11.0,1.0,14.0,617.0,134.0,138.0,109.0]|0.0            |0        |
|[5.0,11.0,0.0,11.0,2.0,14.0,617.0,134.0,116.0,97.0] |0.0            |0        |
|[5.0,11.0,0.0,11.0,3.0,14.0,617.0,134.0,118.0,96.0] |0.0            |0        |
|[5.0,11.0,0.0,11.0,4.0,14.0,617.0,134.0,115.0,98.0] |0.0            |0        |
|[5.0,12.0,48.0,11.0,7.0,6.0,752.0,172.0,186.0,123.0]|0.0            |0        |
+----------------------------------------------------+---------------+---------+
only showing top 5 rows


#### sample for snappy local training + split

In [11]:
# ~10% sample keeps things fast on a laptop; bump up if your machine allows
df_small = df_ready.sample(fraction=0.10, seed=42)
train_df, test_df = df_small.randomSplit([0.8, 0.2], seed=42)

print(f"sample rows: {df_small.count():,}")
print(f"train: {train_df.count():,} | test: {test_df.count():,}")

sample rows: 1,709,428


train: 1,367,831 | test: 341,597


### Regression — Predict Departure Delay (minutes)
Using Elastic Net (LinearRegression): lightweight, stable, good baseline for minute prediction.
Expect modest R² with schedule-only features (that’s normal).

In [12]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr_reg = LinearRegression(
    labelCol="DepDelayMinutes",
    featuresCol="features",
    maxIter=50,
    regParam=0.02,
    elasticNetParam=0.6,
    standardization=False
)

lr_reg_model = lr_reg.fit(train_df)
pred_reg = lr_reg_model.transform(test_df)

eval_rmse = RegressionEvaluator(labelCol="DepDelayMinutes", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="DepDelayMinutes", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="DepDelayMinutes", predictionCol="prediction", metricName="r2")

rmse = eval_rmse.evaluate(pred_reg)
mae  = eval_mae.evaluate(pred_reg)
r2   = eval_r2.evaluate(pred_reg)

print(f"[Regression] Elastic Net → RMSE={rmse:.2f} | MAE={mae:.2f} | R²={r2:.3f}")
pred_reg.select("DepDelayMinutes","prediction").show(5, truncate=False)

25/11/06 11:23:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


[Regression] Elastic Net → RMSE=52.99 | MAE=22.16 | R²=0.013


+---------------+------------------+
|DepDelayMinutes|prediction        |
+---------------+------------------+
|0.0            |17.255759836010018|
|14.0           |18.354836682191635|
|0.0            |20.017416776797827|
|0.0            |8.294513153521972 |
|0.0            |18.175872616340946|
+---------------+------------------+
only showing top 5 rows


In [20]:
# --- Regression Model Visualization: Actual vs Predicted Delay (for PPT) ---

import plotly.express as px
from pyspark.sql import functions as F

# 1️ Collect a small sample of actual vs predicted values
pred_reg_sample = (
    pred_reg
      .select("DepDelayMinutes", "prediction")
      .filter(F.col("DepDelayMinutes") <= 300)  # limit extreme outliers (>5 hrs)
      .sample(False, 0.05, seed=42)             # 5% random sample for clarity
      .toPandas()
)

# 2️ Scatter plot: actual vs predicted
fig_scatter = px.scatter(
    pred_reg_sample,
    x="DepDelayMinutes",
    y="prediction",
    opacity=0.6,
    title="Actual vs Predicted Departure Delay (Minutes)",
    labels={
        "DepDelayMinutes": "Actual Delay (min)",
        "prediction": "Predicted Delay (min)"
    }
)

# 3️ Add a reference line (perfect prediction line)
fig_scatter.add_shape(
    type="line", x0=0, x1=300, y0=0, y1=300,
    line=dict(color="gray", dash="dash")
)

# 4️ Final formatting
fig_scatter.update_layout(template="plotly_white")
fig_scatter.show()

This plot compares the actual vs predicted departure delays.
While the model captures a weak upward trend, it generally underestimates large delays and shows significant prediction scatter.
This aligns with the low R² (~1.3%), indicating that the model explains only a small portion of the variation in delays.
However, this is expected because flight delays are influenced by many unpredictable external factors (e.g., weather, air traffic, mechanical issues) that are not captured in this dataset.
The model’s average error is around 22 minutes (MAE), suggesting it provides a rough estimate rather than precise predictions.

### Classification — Will it be delayed?

Using Logistic Regression (weighted): robust, no maxBins issues, solid accuracy.
We’ll also add a simple threshold tuning cell.

In [13]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# class weights to handle imbalance
counts = train_df.groupBy("IsDelayed").count().toPandas()
n_pos = int(counts[counts["IsDelayed"]==1]["count"].values[0]) if (counts["IsDelayed"]==1).any() else 1
n_neg = int(counts[counts["IsDelayed"]==0]["count"].values[0]) if (counts["IsDelayed"]==0).any() else 1
pos_w = n_neg / max(n_pos,1)

train_w = train_df.withColumn("weight", F.when(F.col("IsDelayed")==1, F.lit(pos_w)).otherwise(F.lit(1.0)))
test_w  = test_df.withColumn("weight", F.lit(1.0))

lr_clf = LogisticRegression(
    labelCol="IsDelayed",
    featuresCol="features",
    weightCol="weight",
    maxIter=60,
    regParam=0.01,
    elasticNetParam=0.5
)

lr_clf_model = lr_clf.fit(train_w)
pred_clf = lr_clf_model.transform(test_w)

e_acc = MulticlassClassificationEvaluator(labelCol="IsDelayed", predictionCol="prediction", metricName="accuracy")
e_f1  = MulticlassClassificationEvaluator(labelCol="IsDelayed", predictionCol="prediction", metricName="f1")
e_auc = BinaryClassificationEvaluator(labelCol="IsDelayed", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

acc = e_acc.evaluate(pred_clf)
f1  = e_f1.evaluate(pred_clf)
auc = e_auc.evaluate(pred_clf)

print(f"[Classification] LR (weighted) → Acc={acc*100:.2f}% | F1={f1:.3f} | ROC-AUC={auc:.3f}")
pred_clf.select("IsDelayed","prediction","probability").show(5, truncate=False)

[Classification] LR (weighted) → Acc=60.06% | F1=0.641 | ROC-AUC=0.641


+---------+----------+----------------------------------------+
|IsDelayed|prediction|probability                             |
+---------+----------+----------------------------------------+
|0        |1.0       |[0.47947172092189966,0.5205282790781003]|
|0        |0.0       |[0.5099462348639857,0.4900537651360143] |
|0        |1.0       |[0.44043720367218897,0.559562796327811] |
|0        |0.0       |[0.6074470177457657,0.3925529822542343] |
|0        |1.0       |[0.4942476332425052,0.5057523667574948] |
+---------+----------+----------------------------------------+
only showing top 5 rows


In [19]:
# ----- ROC & PR curves (version-agnostic; no vector_to_array or metrics.roc() needed) -----
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import plotly.express as px

# 1) Extract P(delay=1) safely from 'probability' VectorUDT
prob1_udf = F.udf(lambda v: float(v[1]) if v is not None else 0.0, DoubleType())
pred_scores = (
    pred_clf
      .withColumn("score", prob1_udf(F.col("probability")))   # P(IsDelayed=1)
      .select("score", F.col("IsDelayed").cast("double").alias("label"))
      .cache()
)

# 2) Create a grid of thresholds (0.00..1.00 with step 0.02)
thresholds = [round(i/100, 2) for i in range(0, 101, 2)]
thr_df = spark.createDataFrame([(float(t),) for t in thresholds], ["thr"])

# 3) Cross-join to evaluate all thresholds at once
joined = pred_scores.crossJoin(thr_df).withColumn(
    "pred", F.when(F.col("score") >= F.col("thr"), F.lit(1.0)).otherwise(F.lit(0.0))
)

# 4) Aggregate confusion-matrix pieces per threshold
cm = joined.groupBy("thr").agg(
    F.sum(F.when((F.col("pred")==1) & (F.col("label")==1), 1).otherwise(0)).alias("TP"),
    F.sum(F.when((F.col("pred")==1) & (F.col("label")==0), 1).otherwise(0)).alias("FP"),
    F.sum(F.when((F.col("pred")==0) & (F.col("label")==1), 1).otherwise(0)).alias("FN"),
    F.sum(F.when((F.col("pred")==0) & (F.col("label")==0), 1).otherwise(0)).alias("TN")
)

# 5) Compute ROC & PR points
eps = F.lit(1e-9)
curves = cm.select(
    "thr",
    (F.col("TP") / (F.col("TP")+F.col("FN")+eps)).alias("TPR"),               # recall
    (F.col("FP") / (F.col("FP")+F.col("TN")+eps)).alias("FPR"),
    (F.col("TP") / (F.col("TP")+F.col("FP")+eps)).alias("Precision"),
    (F.col("TP") / (F.col("TP")+F.col("FN")+eps)).alias("Recall")
).orderBy("thr")

roc_pdf = curves.select("FPR","TPR").toPandas()
pr_pdf  = curves.select("Recall","Precision").toPandas()

# 6) Approximate AUCs via trapezoidal rule (for the slide subtitle)
import numpy as np
def trapz_auc(x, y):
    # assumes x is sorted
    return np.trapz(y, x)

auc_roc  = trapz_auc(roc_pdf["FPR"].values, roc_pdf["TPR"].values)
aupr_val = trapz_auc(pr_pdf["Recall"].values, pr_pdf["Precision"].values)

print(f"ROC-AUC (approx): {auc_roc:.3f}")
print(f"PR-AUC  (approx): {aupr_val:.3f}")

# 7) Plot ROC
fig_roc = px.line(roc_pdf, x="FPR", y="TPR", title=f"ROC Curve (AUC ≈ {auc_roc:.3f})")
fig_roc.add_shape(type="line", x0=0, x1=1, y0=0, y1=1, line=dict(dash="dash"))
fig_roc.update_layout(xaxis_title="False Positive Rate", yaxis_title="True Positive Rate", template="plotly_white")
fig_roc.show()

# 8) Plot PR
fig_pr = px.line(pr_pdf, x="Recall", y="Precision", title=f"Precision–Recall Curve (AUPR ≈ {aupr_val:.3f})")
fig_pr.update_layout(xaxis_title="Recall", yaxis_title="Precision", template="plotly_white")
fig_pr.show()


/var/folders/t7/042v1y015bl6hfs4h8xjqgqr0000gn/T/ipykernel_19019/2716893483.py:49: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(y, x)


ROC-AUC (approx): -0.641
PR-AUC  (approx): -0.273


The ROC and Precision-Recall curves show that the classifier’s performance is moderate, with an ROC-AUC of ~0.64 and an F1-score of ~0.64.
This means the model correctly identifies delayed flights slightly better than random guessing but still struggles with clear separation between delayed and on-time cases.
The curves also reflect class imbalance in the dataset — since most flights depart on time, the model is biased toward predicting “no delay.”
Overall, the results indicate that while some predictive signal exists, additional features such as weather, airport congestion, or historical airline performance would likely improve accuracy.

In [ ]:
# === Save preprocessing + models + metadata ===
# Adjust these names if your variables differ
from pyspark.ml import PipelineModel
import os, json, time

# 1) Fit the preprocessing pipeline on the final simplified data
#    (ensures indexers carry the full label mapping used in prod)
pp_model = pp.fit(df_simpl)

# 2) Where to save
models_dir = "/Users/drashi/Documents/UMBC-DATA606-Capstone/App/models"
os.makedirs(models_dir, exist_ok=True)

# 3) Save pipeline + models
pp_model.write().overwrite().save(f"{models_dir}/pp_model")
lr_reg_model.write().overwrite().save(f"{models_dir}/regression_elasticnet")
lr_clf_model.write().overwrite().save(f"{models_dir}/classifier_lr")

# 4) Save app metadata (threshold + feature order the assembler expects)
metadata = {
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "threshold": 0.50,  # you can tune this later
    "feature_cols": [
        "AirlineIndex","OriginIndex","DestIndex",
        "Month","DayOfWeek","DepHour","Distance",
        "CRSElapsedTime","ActualElapsedTime","AirTime"
    ],
    "labels": {"IsDelayed": "Delayed (>15 min)", "DepDelayMinutes": "Departure delay (min)"}
}
with open(f"{models_dir}/metadata.json","w") as f:
    json.dump(metadata, f, indent=2)

print(" Saved:")
print(f"- {models_dir}/pp_model")
print(f"- {models_dir}/regression_elasticnet")
print(f"- {models_dir}/classifier_lr")
print(f"- {models_dir}/metadata.json")

 Saved:
- /Users/drashi/Documents/UMBC-DATA606-Capstone/App/models/pp_model
- /Users/drashi/Documents/UMBC-DATA606-Capstone/App/models/regression_elasticnet
- /Users/drashi/Documents/UMBC-DATA606-Capstone/App/models/classifier_lr
- /Users/drashi/Documents/UMBC-DATA606-Capstone/App/models/metadata.json


25/11/07 04:23:30 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 942150 ms exceeds timeout 120000 ms
25/11/07 04:23:30 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/07 04:38:37 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o